# Membuat SparkSession & Dataset
Jalankan cell ini terlebih dahulu di setiap sesi Jupyter baru.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/03 21:21:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


## Data CSV

In [3]:
# Membuat dataset yang lebih besar (20.000 baris) khusus untuk latihan ini
np.random.seed(11)
n = 20000
data_besar = {
    "order_id": [f"O{i}" for i in range(n)],
    "customer_id": np.random.randint(1, 21, size=n),
    "kategori": np.random.choice(["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"], size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo", "Yogyakarta", "Purworejo"], size=n),
    "metode_pembayaran": np.random.choice(["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
pd.DataFrame(data_besar).to_csv("data_transaksi_besar.csv", index=False)

csv_size = os.path.getsize("data_transaksi_besar.csv")
print(f"Ukuran CSV: {csv_size:,} bytes ({csv_size/1024:.1f} KB) untuk {n:,} baris")

Ukuran CSV: 892,909 bytes (872.0 KB) untuk 20,000 baris


In [4]:
# Membaca CSV, lalu menuliskannya ulang sebagai Parquet
df_besar = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_besar.write.mode("overwrite").parquet("data_transaksi_besar_parquet")

# Menghitung total ukuran seluruh berkas .parquet yang dihasilkan
parquet_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fn in os.walk("data_transaksi_besar_parquet")
    for f in fn if f.endswith(".parquet")
)

print(f"Ukuran CSV      : {csv_size:,} bytes ({csv_size/1024:.1f} KB)")
print(f"Ukuran Parquet  : {parquet_size:,} bytes ({parquet_size/1024:.1f} KB)")
print(f"Parquet {(1 - parquet_size/csv_size)*100:.1f}% lebih kecil dibanding CSV")

[Stage 2:>                                                          (0 + 1) / 1]

Ukuran CSV      : 892,909 bytes (872.0 KB)
Ukuran Parquet  : 202,197 bytes (197.5 KB)
Parquet 77.4% lebih kecil dibanding CSV


## Data JSON

In [5]:
import json

# Membuat data pelanggan sebagai JSON Lines (satu objek JSON per baris — format umum untuk big data)
np.random.seed(21)
pelanggan = [
    {
        "customer_id": i,
        "nama": f"Pelanggan{i}",
        "membership": np.random.choice(["Silver", "Gold", "Platinum"]),
        "kota_domisili": np.random.choice(["Magelang", "Semarang", "Solo", "Yogyakarta", "Purworejo"])
    }
    for i in range(1, 21)
]

with open("data_pelanggan.json", "w") as f:
    for p in pelanggan:
        f.write(json.dumps(p) + "\n")

print("Berkas data_pelanggan.json berhasil dibuat (format JSON Lines).")

Berkas data_pelanggan.json berhasil dibuat (format JSON Lines).


# Latihan Mandiri

In [6]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan6").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi_etl = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_pelanggan_etl = spark.read.json("data_pelanggan.json")
print("Siap untuk latihan.")

Siap untuk latihan.


#### SOAL 1

In [7]:
# SOAL 1
# Skema dari Parquet
df_parquet_latihan = spark.read.parquet("data_transaksi_besar_parquet")
print("Skema dari Parquet:")
df_parquet_latihan.printSchema()

# Skema dari CSV (inferSchema=True)
print("Skema dari CSV:")
df_transaksi_etl.printSchema()

Skema dari Parquet:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)

Skema dari CSV:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)



#### SOAL 2

In [8]:
# Soal 2
df_gabung = df_transaksi_etl.join(df_pelanggan_etl, on="customer_id", how="left")
df_gabung.groupBy("membership").count().show()

+----------+-----+
|membership|count|
+----------+-----+
|  Platinum| 4016|
|    Silver| 8994|
|      Gold| 6990|
+----------+-----+



#### SOAL 3

In [9]:
# Soal 3
df_gabung.write.mode("overwrite").partitionBy("membership").parquet("data_transaksi_membership")
!ls data_transaksi_membership

[Stage 12:>                                                         (0 + 1) / 1]

'membership=Gold'  'membership=Platinum'  'membership=Silver'   _SUCCESS


#### SOAL 4
Soal 4 menanyakan mengapa partitioning berdasarkan kota cocok untuk skenario kita, tetapi menjadi pilihan buruk jika dipakai pada **order_id**. Jawabannya seperti ini :

*(Partitioning berdasarkan kota cocok karena kolom itu hanya punya 5 nilai berbeda, jadi Spark hanya membuat 5 folder yang masing-masing berisi banyak baris, dan kueri yang menyaring satu kota cukup membuka satu folder saja (partition pruning). Kalau dipakai pada order_id yang nilainya selalu unik, Spark akan membuat satu folder untuk setiap baris, yaitu 20.000 folder dengan isi satu baris saja. Akibatnya muncul puluhan ribu file kecil yang membebani pengelolaan metadata dan membuat proses baca dan tulis lambat, padahal partisi seharusnya dipakai pada kolom dengan jumlah nilai unik yang sedikit.)*

# TUGAS MANDIRI

#### Menyiapkan Dataset
*Jalankan cell berikut untuk menghasilkan tiga sumber data sekaligus.*

In [10]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


### A. EXTRACT 

In [11]:
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk    = spark.read.json("tugas6_produk.json")
df_ulasan    = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print("=== TRANSAKSI ===")
print("Jumlah baris:", df_transaksi.count())
df_transaksi.printSchema()

print("=== PRODUK ===")
print("Jumlah baris:", df_produk.count())
df_produk.printSchema()

print("=== ULASAN ===")
print("Jumlah baris:", df_ulasan.count())
df_ulasan.printSchema()

=== TRANSAKSI ===
Jumlah baris: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

=== PRODUK ===
Jumlah baris: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

=== ULASAN ===
Jumlah baris: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



### B. TRANSFORM — Penggabungan 

In [12]:
# Transaksi > Ulasan LEFT JOIN
df_gabung = df_transaksi.join(df_ulasan, on="order_id", how="left")

# Transaksi > Produk INNER JOIN
df_gabung = df_gabung.join(df_produk, on="product_id", how="inner")

# total pendapatan
df_gabung = df_gabung.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

print("Jumlah baris setelah join:", df_gabung.count())
df_gabung.select("order_id", "product_id", "kategori", "unit_terjual", "harga", "total_pendapatan", "rating").show(5)

Jumlah baris setelah join: 5000
+--------+----------+------------+------------+------+----------------+------+
|order_id|product_id|    kategori|unit_terjual| harga|total_pendapatan|rating|
+--------+----------+------------+------------+------+----------------+------+
|     TX0|        21|  Elektronik|           2| 25000|           50000|     1|
|     TX1|         8|     Fashion|           6|250000|         1500000|  NULL|
|     TX2|        25|  Elektronik|           4| 25000|          100000|     2|
|     TX3|         3|     Fashion|           4| 75000|          300000|     5|
|     TX4|        19|Rumah Tangga|           6| 75000|          450000|  NULL|
+--------+----------+------------+------------+------+----------------+------+
only showing top 5 rows



### C. TRANSFORM — Penanganan Data Kosong & Pengayaan

In [13]:
# Kolom ada_ulasan: dihitung otomatis dari rating
df_gabung = df_gabung.withColumn("ada_ulasan", col("rating").isNotNull())

# Isi rating kosong dengan 0 (SETELAH ada_ulasan dibuat)
df_etl = df_gabung.na.fill(0, subset=["rating"])

# Pengecekan
print("Rating yang masih null :", df_etl.filter(col("rating").isNull()).count())
print("Transaksi dengan ulasan:", df_etl.filter(col("ada_ulasan") == True).count())
print("Transaksi tanpa ulasan :", df_etl.filter(col("ada_ulasan") == False).count())

df_etl.select("order_id", "kategori", "rating", "ada_ulasan").show(5)

Rating yang masih null : 0
Transaksi dengan ulasan: 3500
Transaksi tanpa ulasan : 1500
+--------+------------+------+----------+
|order_id|    kategori|rating|ada_ulasan|
+--------+------------+------+----------+
|     TX0|  Elektronik|     1|      true|
|     TX1|     Fashion|     0|     false|
|     TX2|  Elektronik|     2|      true|
|     TX3|     Fashion|     5|      true|
|     TX4|Rumah Tangga|     0|     false|
+--------+------------+------+----------+
only showing top 5 rows



### D. LOAD

In [14]:
!hdfs dfs -mkdir -p /user/xiuviu/tugas6

df_etl.write.mode("overwrite").partitionBy("kategori").parquet("hdfs://localhost:9000/user/xiuviu/tugas6/hasil_etl")

print("Load selesai. Hasil tersimpan di HDFS.")

[Stage 52:>                                                         (0 + 1) / 1]

Load selesai. Hasil tersimpan di HDFS.


In [16]:
# Verifikasi struktur folder di HDFS
!hdfs dfs -ls -R /user/xiuviu/tugas6/hasil_etl

-rw-r--r--   3 xiuviu supergroup          0 2026-10-03 22:08 /user/xiuviu/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - xiuviu supergroup          0 2026-10-03 22:08 /user/xiuviu/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 xiuviu supergroup      16366 2026-10-03 22:08 /user/xiuviu/tugas6/hasil_etl/kategori=Elektronik/part-00000-c272fa81-0d2e-4d58-93e9-3af8a008b6f2.c000.snappy.parquet
drwxr-xr-x   - xiuviu supergroup          0 2026-10-03 22:08 /user/xiuviu/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 xiuviu supergroup      11133 2026-10-03 22:08 /user/xiuviu/tugas6/hasil_etl/kategori=Fashion/part-00000-c272fa81-0d2e-4d58-93e9-3af8a008b6f2.c000.snappy.parquet
drwxr-xr-x   - xiuviu supergroup          0 2026-10-03 22:08 /user/xiuviu/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 xiuviu supergroup      10392 2026-10-03 22:08 /user/xiuviu/tugas6/hasil_etl/kategori=Kesehatan/part-00000-c272fa81-0d2e-4d58-93e9-3af8a008b6f2.c000.snappy.parquet
drwxr-xr-x   - xiuviu supergroup      

In [18]:
# Baca kembali dari HDFS sebagai bukti data tersimpan utuh
df_final = spark.read.parquet("hdfs://localhost:9000/user/xiuviu/tugas6/hasil_etl")

print("Jumlah baris hasil akhir      :", df_final.count())
print("Jumlah baris sebelum disimpan :", df_etl.count())
df_final.printSchema()

Jumlah baris hasil akhir      : 5000
Jumlah baris sebelum disimpan : 5000
root
 |-- product_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- rating: integer (nullable = true)
 |-- harga: long (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- total_pendapatan: long (nullable = true)
 |-- ada_ulasan: boolean (nullable = true)
 |-- kategori: string (nullable = true)



### E. Insight Akhir

In [23]:
from pyspark.sql.functions import round

df_insight = df_final.groupBy("kategori").agg(
    count("order_id").alias("jumlah_transaksi"),
    round(avg(col("ada_ulasan").cast("int")) * 100, 2).alias("persen_ada_ulasan")
).orderBy("persen_ada_ulasan")

df_insight.show()

+------------+----------------+-----------------+
|    kategori|jumlah_transaksi|persen_ada_ulasan|
+------------+----------------+-----------------+
|     Makanan|             536|            68.84|
|   Kesehatan|             961|            68.89|
|     Fashion|            1035|            70.14|
|Rumah Tangga|             815|            70.55|
|  Elektronik|            1653|            70.66|
+------------+----------------+-----------------+



In [25]:
df_final.createOrReplaceTempView("hasil_etl")

spark.sql("""
    SELECT kategori,
           COUNT(order_id) AS jumlah_transaksi,
           ROUND(AVG(CAST(ada_ulasan AS INT)) * 100, 2) AS persen_ada_ulasan
    FROM hasil_etl
    GROUP BY kategori
    ORDER BY persen_ada_ulasan ASC
""").show()

+------------+----------------+-----------------+
|    kategori|jumlah_transaksi|persen_ada_ulasan|
+------------+----------------+-----------------+
|     Makanan|             536|            68.84|
|   Kesehatan|             961|            68.89|
|     Fashion|            1035|            70.14|
|Rumah Tangga|             815|            70.55|
|  Elektronik|            1653|            70.66|
+------------+----------------+-----------------+



In [27]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.


**PENGUJIAN**

*Untuk memastikan pipeline berjalan dengan baik, ada tiga hal yang perlu dicek. Pertama, jumlah data setelah proses join tetap 5.000 baris, sama seperti jumlah transaksi awal. Kedua, kolom ada_ulasan harus sesuai dengan data asli, yaitu 3.500 transaksi memiliki ulasan dan 1.500 tidak, serta tidak ada lagi nilai rating yang null. Terakhir, data yang sudah disimpan di HDFS harus bisa dibaca kembali dan jumlah barisnya tetap sama seperti sebelum disimpan.*